# 04 — Voto exterior (DE 27)

Análisis focalizado del voto de peruanos residentes en el extranjero: distrito electoral 27, `idAmbitoGeografico=2`. ~2,543 mesas × 5 elecciones distribuidas en 210 ciudades internacionales.

In [ ]:
import polars as pl

cab = pl.read_parquet('parquet/actas_cabecera.parquet')
ext = cab.filter(pl.col('idAmbitoGeografico') == 2)
print(f'actas ámbito exterior: {ext.shape[0]:,}')
print(f'mesas únicas exterior: {ext["idMesaRef"].n_unique():,}')
ext.group_by('idEleccion').agg(pl.len().alias('actas'))

In [ ]:
# Estado del conteo en exterior
ext.group_by('codigoEstadoActa').agg(
    pl.len().alias('actas'),
    (pl.len() * 100.0 / ext.shape[0]).round(2).alias('pct')
).sort('actas', descending=True)

In [ ]:
# Top 10 países con más electores hábiles
pres_ext = ext.filter(pl.col('idEleccion') == 10)
por_pais = pres_ext.group_by('ubigeoDepartamento').agg([
    pl.col('totalElectoresHabiles').sum().alias('habiles'),
    pl.len().alias('mesas'),
])
# ubigeoDepartamento exterior son códigos 91XXXX-95XXXX (91=África, 92=América, etc.)
por_pais.sort('habiles', descending=True).head(10)

In [ ]:
# Cruzar con padrón RENIEC por país
padron_ext = pl.read_parquet('parquet/dim_padron.parquet').filter(pl.col('residencia') == 'Extranjero')
padron_ext.sort('total_electores', descending=True).head(10).select([
    'pais_nombre', 'total_electores', 'hombres', 'mujeres'
])

In [ ]:
# Tasa de instalación — cuántas de las 2,543 mesas exterior reportan actas C
pres_ext_c = pres_ext.filter(pl.col('codigoEstadoActa') == 'C')
total_mesas_exterior = pres_ext['idMesaRef'].n_unique()
mesas_c = pres_ext_c['idMesaRef'].n_unique()
print(f'mesas exterior total: {total_mesas_exterior:,}')
print(f'mesas con acta C Presidencial: {mesas_c:,}')
print(f'tasa instalación: {mesas_c * 100.0 / total_mesas_exterior:.2f}%')